# 生成经验回放数据（防止灾难性遗忘）

用 Qwen3-4B 自身生成 30 条通用问答数据，微调时混入训练集，让模型不会忘记基础能力。

**原理：** 微调时只训练角色扮演数据会导致模型遗忘通用知识。混入一批由模型自己生成的通用问答，相当于告诉模型「这些回答我以前就会，继续保持」。

In [ ]:
! pip install -q -U transformers bitsandbytes accelerate

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"GPU可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 加载模型

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    print("pad_token 为空，设置为 eos_token")
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("模型加载完成")

## 定义提示词（10类 × 3条 = 30条）

问题保持简短，回答在 150 token 以内即可覆盖。

In [ ]:
# 10个话题类别，每类3个提示，共30条
# 问题简短，确保回答在150 token内

PROMPTS = [
    # --- 1. 日常知识 ---
    "为什么天空是蓝色的？",
    "人一天需要喝多少水？",
    "为什么打哈欠会传染？",

    # --- 2. 科普常识 ---
    "光从太阳到地球需要多长时间？",
    "为什么夏天虫子特别多？",
    "月亮为什么有时候看起来特别大？",

    # --- 3. 历史文化 ---
    "中国的四大发明是哪四个？",
    "为什么春节要贴春联、放鞭炮？",
    "丝绸之路是什么？",

    # --- 4. 中文表达 ---
    "解释一下'塞翁失马，焉知非福'这句话",
    "用三个词描述一个秋天的傍晚",
    "'的地得'分别在什么情况下使用？",

    # --- 5. 生活技巧 ---
    "衣服上沾了红酒怎么快速处理？",
    "手机充电有什么保护电池的小技巧？",
    "眼睛疲劳时如何快速缓解？",

    # --- 6. 健康饮食 ---
    "早餐吃什么比较健康又方便？",
    "感冒初期应该怎么处理？",
    "久坐一天，有什么简单的拉伸动作推荐？",

    # --- 7. 职场学习 ---
    "番茄工作法是什么？怎么用？",
    "面试时被问'你最大的缺点'怎么回答比较好？",
    "如何快速记会议要点？",

    # --- 8. 情感人际 ---
    "朋友情绪低落时，我该怎么安慰他？",
    "如何礼貌地拒绝别人的请求？",
    "如何在新环境中快速融入团队？",

    # --- 9. 简单数学 ---
    "正方形面积是64平方厘米，周长是多少？",
    "鸡兔同笼：头共10个，脚共28只，鸡和兔各有几只？",
    "一个人花30元买了一只鸡，40元卖出，50元买回，60元卖出，赚了多少？",

    # --- 10. 创意趣味 ---
    "100元以内，送什么礼物能让朋友感到惊喜？",
    "如何把每天的通勤时间变得更有价值？",
    "推荐一件可以让日常生活更有趣的小事",
]

print(f"共 {len(PROMPTS)} 个提示词，涵盖 10 个类别")

# 批量生成回答

In [ ]:
def generate_response(prompt: str, max_new_tokens: int = 150) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    return response.strip()


# 生成所有回答
replay_data = []
failed = []

print(f"开始生成，共 {len(PROMPTS)} 条...\n")

for i, prompt in enumerate(PROMPTS):
    try:
        response = generate_response(prompt)
        if len(response.strip()) < 5:
            raise ValueError(f"回答过短: {repr(response)}")

        replay_data.append({
            "conversations": [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": response},
            ]
        })

        # 每5条打印一次进度
        if (i + 1) % 5 == 0 or i == 0:
            print(f"[{i+1}/{len(PROMPTS)}] Q: {prompt}")
            print(f"         A: {response[:80]}{'...' if len(response) > 80 else ''}")
            print()

    except Exception as e:
        print(f"[{i+1}] 失败: {prompt} 错误: {e}")
        failed.append(i)

print(f"\n生成完成：成功 {len(replay_data)} 条，失败 {len(failed)} 条")

# 验证质量

In [ ]:
# 抽查几条，人工确认质量
import random

samples = random.sample(replay_data, min(5, len(replay_data)))
print("=== 随机抽查 5 条 ===")
for s in samples:
    q = s["conversations"][0]["content"]
    a = s["conversations"][1]["content"]
    print(f"Q: {q}")
    print(f"A: {a[:200]}{'...' if len(a) > 200 else ''}")
    print("-" * 60)

# 保存数据

In [ ]:
OUTPUT_FILE = "replay_data.json"

# 保存为和 train_data_augmented.json 兼容的格式
# system_prompt 留空，微调时不注入角色 prompt，让模型保持通用能力
output = {
    "system_prompt": "",
    "conversations": replay_data,
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"已保存 {len(replay_data)} 条数据到 {OUTPUT_FILE}")
print(f"文件大小: {__import__('os').path.getsize(OUTPUT_FILE) / 1024:.1f} KB")

In [ ]:
# 下载到本地
from google.colab import files
files.download(OUTPUT_FILE)
print(f"已下载 {OUTPUT_FILE}")